# Student-Accessible Plagiarism Detection Portal

## Assignment No. 3 – End-to-End Portal Design

### Project Description

The Student-Accessible Plagiarism Detection Portal is a Python-based system designed to help university students check the similarity of their assignments.

The system allows students to register, log in, submit assignments, and compare their work with previously submitted assignments.

It calculates a similarity percentage and displays matching sentences to help students identify possible similarities.

The system uses Python functions, loops, JSON file handling, and exception handling to manage student records and generate reports.


In [1]:
import json
import os
import re
import hashlib
from datetime import datetime

print("Student Plagiarism Detection Portal")
print("Libraries imported successfully!")

Student Plagiarism Detection Portal
Libraries imported successfully!


In [3]:
DATA_DIR = "data"

USERS_FILE = os.path.join(DATA_DIR, "users.json")
SUBMISSIONS_FILE = os.path.join(DATA_DIR, "submissions.json")
REPORTS_FILE = os.path.join(DATA_DIR, "reports.json")


def ensure_data_files():

    os.makedirs(DATA_DIR, exist_ok=True)

    for file_path in [USERS_FILE, SUBMISSIONS_FILE, REPORTS_FILE]:

        if not os.path.exists(file_path):

            with open(file_path, "w") as file:
                json.dump([], file, indent=4)


ensure_data_files()

print("Data files created successfully!")

Data files created successfully!


In [4]:
def load_records(file_path):

    try:

        with open(file_path, "r") as file:
            records = json.load(file)

        return records

    except FileNotFoundError:

        print("File not found!")
        return []

    except json.JSONDecodeError:

        print("Invalid file format!")
        return []

    except PermissionError:

        print("Permission denied!")
        return []

    finally:

        print("File reading operation completed.")


def save_records(file_path, records):

    try:

        with open(file_path, "w") as file:
            json.dump(records, file, indent=4)

        return True

    except (PermissionError, OSError) as error:

        print("Error saving file:", error)
        return False

    finally:

        print("File saving operation completed.")

In [5]:
def register_user():

    users = load_records(USERS_FILE)

    print("\n--- Student Registration ---")

    username = input("Enter a username: ").strip()

    if username == "":
        print("Username cannot be empty.")
        return

    # Check whether the username already exists
    for user in users:

        if user["username"].lower() == username.lower():
            print("This username is already registered.")
            return

    password = input("Create a password: ")

    if password == "":
        print("Password cannot be empty.")
        return

    # Hash the password
    hashed_password = hashlib.sha256(password.encode()).hexdigest()

    new_user = {
        "username": username,
        "password": hashed_password,
        "registered_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    users.append(new_user)

    if save_records(USERS_FILE, users):
        print("Registration successful!")

In [6]:
def user_login():

    users = load_records(USERS_FILE)

    print("\n--- Student Login ---")

    username = input("Enter your username: ").strip()
    password = input("Enter your password: ")

    hashed_password = hashlib.sha256(password.encode()).hexdigest()

    for user in users:

        if (user["username"].lower() == username.lower()
                and user["password"] == hashed_password):

            print("\nLogin successful!")
            print("Welcome,", user["username"])

            return user["username"]

    print("Invalid username or password.")
    return None

In [7]:
def add_record(username):

    submissions = load_records(SUBMISSIONS_FILE)

    print("\n--- Submit Assignment ---")

    title = input("Enter your assignment title: ").strip()

    if title == "":
        print("Assignment title cannot be empty.")
        return

    print("\nEnter your assignment text.")
    print("Type END on a separate line when you have finished.\n")

    lines = []

    while True:

        line = input()

        if line.strip().upper() == "END":
            break

        lines.append(line)

    content = "\n".join(lines).strip()

    if content == "":
        print("Assignment cannot be empty.")
        return

    # Generate a unique submission ID
    submission_id = max(
        (record.get("submission_id", 0) for record in submissions),
        default=0
    ) + 1

    new_submission = {
        "submission_id": submission_id,
        "username": username,
        "title": title,
        "content": content,
        "submitted_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    submissions.append(new_submission)

    if save_records(SUBMISSIONS_FILE, submissions):

        print("\nAssignment submitted successfully!")
        print("Your submission ID is:", submission_id)

In [9]:
register_user()

File reading operation completed.

--- Student Registration ---


Enter a username:  student005
Create a password:  faststudent5


File saving operation completed.
Registration successful!


In [11]:
logged_in_user = user_login()

File reading operation completed.

--- Student Login ---


Enter your username:  student005
Enter your password:  faststudent5



Login successful!
Welcome, student005


In [12]:
if logged_in_user:
    add_record(logged_in_user)
else:
    print("Please log in first.")

File reading operation completed.

--- Submit Assignment ---


Enter your assignment title:  Python Project



Enter your assignment text.
Type END on a separate line when you have finished.



 Python is a high-level programming language.
 It is widely used for software development.
 we will be using python for our project.
 END


File saving operation completed.

Assignment submitted successfully!
Your submission ID is: 4


In [13]:
def clean_text(text):

    # Convert all text to lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [14]:
sample_text = "Python is GREAT! Python is easy to learn."

cleaned_text = clean_text(sample_text)

print("Original Text:")
print(sample_text)

print("\nCleaned Text:")
print(cleaned_text)

Original Text:
Python is GREAT! Python is easy to learn.

Cleaned Text:
python is great python is easy to learn


In [15]:
def calculate_similarity(text1, text2):

    # Clean both assignment texts
    text1 = clean_text(text1)
    text2 = clean_text(text2)

    # Convert the words into sets
    words1 = set(text1.split())
    words2 = set(text2.split())

    # Find common and total unique words
    common_words = words1.intersection(words2)
    total_words = words1.union(words2)

    # Calculate Jaccard similarity
    if len(total_words) == 0:
        return 0

    similarity = (len(common_words) / len(total_words)) * 100

    return round(similarity, 2)

In [17]:
assignment1 = """
Python is a high-level programming language.
It is widely used for software development.
we will be using python for out project.
"""

assignment2 = """
Python is a high-level programming language.
It is widely used for software development.
Python is useful for data analysis.
"""

similarity = calculate_similarity(assignment1, assignment2)

print("Assignment 1:")
print(assignment1)

print("Assignment 2:")
print(assignment2)

print("\nSimilarity Percentage:", similarity, "%")

Assignment 1:

Python is a high-level programming language.
It is widely used for software development.
we will be using python for out project.

Assignment 2:

Python is a high-level programming language.
It is widely used for software development.
Python is useful for data analysis.


Similarity Percentage: 59.09 %


In [18]:
test1 = "Python is a programming language."
test2 = "Python is a programming language."

result = calculate_similarity(test1, test2)

print("Assignment 1:", test1)
print("Assignment 2:", test2)

print("\nSimilarity Percentage:", result, "%")

Assignment 1: Python is a programming language.
Assignment 2: Python is a programming language.

Similarity Percentage: 100.0 %


In [19]:
def compare_submissions(submission_id):

    submissions = load_records(SUBMISSIONS_FILE)

    # Find the assignment we want to check
    target = None

    for record in submissions:

        if record["submission_id"] == submission_id:
            target = record
            break

    if target is None:
        print("Assignment not found.")
        return []

    print("\n--- Assignment Similarity Check ---")
    print("Checking:", target["title"])
    print("Submitted by:", target["username"])

    comparisons = []

    # Compare with assignments from other students
    for record in submissions:

        if record["submission_id"] == submission_id:
            continue

        if record["username"].lower() == target["username"].lower():
            continue

        percentage = calculate_similarity(
            target["content"],
            record["content"]
        )

        comparisons.append({
            "compared_with": record["username"],
            "title": record["title"],
            "similarity": percentage
        })

    if not comparisons:
        print("\nNo assignments from other students are available for comparison.")
        return []

    # Display the comparison results
    comparisons.sort(
        key=lambda item: item["similarity"],
        reverse=True
    )

    for result in comparisons:

        print("\nCompared with:", result["compared_with"])
        print("Assignment:", result["title"])
        print("Similarity:", result["similarity"], "%")

    return comparisons

In [20]:
register_user()

File reading operation completed.

--- Student Registration ---


Enter a username:  student006
Create a password:  faststudent6


File saving operation completed.
Registration successful!


In [21]:
second_user = user_login()

if second_user:
    add_record(second_user)

File reading operation completed.

--- Student Login ---


Enter your username:  student006
Enter your password:  faststudent6



Login successful!
Welcome, student006
File reading operation completed.

--- Submit Assignment ---


Enter your assignment title:  Project for Programming



Enter your assignment text.
Type END on a separate line when you have finished.



 Python is a high-level programming language.
 It is widely used for software development.
 We will be using it for our project.
 END


File saving operation completed.

Assignment submitted successfully!
Your submission ID is: 5


In [22]:
def find_matching_sentences(text1, text2):

    # Split both assignments into individual sentences
    sentences1 = re.split(r"[.!?\n]+", text1)
    sentences2 = re.split(r"[.!?\n]+", text2)

    # Store cleaned sentences from the second assignment
    cleaned_sentences2 = set()

    for sentence in sentences2:

        cleaned = clean_text(sentence)

        if cleaned:
            cleaned_sentences2.add(cleaned)

    matching_sentences = []

    # Compare sentences from the first assignment
    for sentence in sentences1:

        cleaned = clean_text(sentence)

        if cleaned and cleaned in cleaned_sentences2:

            if sentence.strip() not in matching_sentences:
                matching_sentences.append(sentence.strip())

    return matching_sentences

In [23]:
submissions = load_records(SUBMISSIONS_FILE)

assignment_one = None
assignment_two = None

for record in submissions:

    if record["submission_id"] == 1:
        assignment_one = record

    elif record["submission_id"] == 2:
        assignment_two = record


if assignment_one and assignment_two:

    matches = find_matching_sentences(
        assignment_one["content"],
        assignment_two["content"]
    )

    print("Matching Sentences Between Assignments")
    print("---------------------------------------")

    if matches:

        for sentence in matches:
            print("-", sentence)

    else:
        print("No matching sentences found.")

else:
    print("Required assignments were not found.")

File reading operation completed.
Matching Sentences Between Assignments
---------------------------------------
- Python is a high-level programming language
- It is widely used for software development


In [24]:
def generate_report(submission_id, comparisons):

    submissions = load_records(SUBMISSIONS_FILE)
    reports = load_records(REPORTS_FILE)

    # Find the submitted assignment
    target = None

    for record in submissions:

        if record["submission_id"] == submission_id:
            target = record
            break

    if target is None:
        print("Assignment not found.")
        return

    # Find matching sentences for each comparison
    comparison_details = []

    for result in comparisons:

        other_submission = None

        for record in submissions:

            if (record["username"] == result["compared_with"]
                    and record["title"] == result["title"]):

                other_submission = record
                break

        if other_submission:

            matches = find_matching_sentences(
                target["content"],
                other_submission["content"]
            )

            comparison_details.append({
                "compared_with": result["compared_with"],
                "assignment_title": result["title"],
                "similarity_percentage": result["similarity"],
                "matching_sentences": matches
            })

    report = {
        "report_id": len(reports) + 1,
        "submission_id": submission_id,
        "student": target["username"],
        "assignment_title": target["title"],
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "comparisons": comparison_details
    }

    reports.append(report)

    if save_records(REPORTS_FILE, reports):

        print("\nSimilarity report generated successfully!")
        print("Report ID:", report["report_id"])
        print("Student:", report["student"])
        print("Assignment:", report["assignment_title"])

In [25]:
comparison_results = compare_submissions(1)

generate_report(1, comparison_results)

File reading operation completed.

--- Assignment Similarity Check ---
Checking: Introduction to Python Programming
Submitted by: student 1

Compared with: student005
Assignment: Python Project
Similarity: 40.62 %

Compared with: student006
Assignment: Project for Programming
Similarity: 40.62 %

Compared with: student001
Assignment: Python Project
Similarity: 38.24 %

Compared with: student002
Assignment: Python Assignment
Similarity: 31.43 %
File reading operation completed.
File reading operation completed.
File saving operation completed.

Similarity report generated successfully!
Report ID: 2
Student: student 1
Assignment: Introduction to Python Programming


In [26]:
saved_reports = load_records(REPORTS_FILE)

print("Total Reports Generated:", len(saved_reports))

for report in saved_reports:

    print("\nReport ID:", report["report_id"])
    print("Student:", report["student"])
    print("Assignment:", report["assignment_title"])
    print("Generated At:", report["generated_at"])

File reading operation completed.
Total Reports Generated: 2

Report ID: 1
Student: student 1
Assignment: Introduction to Python Programming
Generated At: 2026-11-14 22:03:17

Report ID: 2
Student: student 1
Assignment: Introduction to Python Programming
Generated At: 2026-11-14 22:54:39


In [27]:
def view_my_reports(username):

    reports = load_records(REPORTS_FILE)

    print("\n--- My Similarity Reports ---")

    found = False

    for report in reports:

        if report["student"].lower() == username.lower():

            found = True

            print("\nReport ID:", report["report_id"])
            print("Assignment:", report["assignment_title"])
            print("Generated:", report["generated_at"])

            for comparison in report["comparisons"]:

                print("Compared with:", comparison["compared_with"])
                print("Similarity:",
                      comparison["similarity_percentage"], "%")

                print("Matching sentences:")

                if comparison["matching_sentences"]:

                    for sentence in comparison["matching_sentences"]:
                        print("-", sentence)

                else:
                    print("No exact matching sentences.")

    if not found:
        print("You have no previous reports.")

In [30]:
view_my_reports("student006")

File reading operation completed.

--- My Similarity Reports ---
You have no previous reports.


In [31]:
def search_record():

    submissions = load_records(SUBMISSIONS_FILE)

    print("\n--- Search Assignments ---")

    keyword = input("Enter student username or assignment title: ").strip().lower()

    if keyword == "":
        print("Please enter a search keyword.")
        return

    found = False

    for record in submissions:

        if (keyword in record["username"].lower()
                or keyword in record["title"].lower()):

            found = True

            print("\nSubmission ID:", record["submission_id"])
            print("Student:", record["username"])
            print("Assignment:", record["title"])
            print("Submitted:", record["submitted_at"])

    if not found:
        print("No matching assignments found.")

In [32]:
search_record()

File reading operation completed.

--- Search Assignments ---


Enter student username or assignment title:  student005



Submission ID: 4
Student: student005
Assignment: Python Project
Submitted: 2026-11-14 22:44:14


In [33]:
def student_menu(username):

    while True:

        print("\n========== STUDENT DASHBOARD ==========")
        print("Welcome,", username)

        print("\n1. Submit Assignment")
        print("2. Check Assignment Similarity")
        print("3. View My Reports")
        print("4. Search Assignments")
        print("5. Logout")

        choice = input("\nEnter your choice (1-5): ")

        if choice == "1":

            add_record(username)

        elif choice == "2":

            try:

                submission_id = int(
                    input("Enter your submission ID: ")
                )

                submissions = load_records(SUBMISSIONS_FILE)

                owned = any(
                    record["submission_id"] == submission_id
                    and record["username"].lower() == username.lower()
                    for record in submissions
                )

                if owned:

                    comparisons = compare_submissions(submission_id)

                    if comparisons:
                        generate_report(submission_id, comparisons)

                else:
                    print("Invalid submission ID or assignment not found.")

            except ValueError:
                print("Please enter a valid numeric ID.")

        elif choice == "3":

            view_my_reports(username)

        elif choice == "4":

            search_record()

        elif choice == "5":

            print("You have been logged out.")
            break

        else:

            print("Invalid choice. Please select from 1 to 5.")

In [34]:
def main():

    print("======================================")
    print(" STUDENT PLAGIARISM DETECTION PORTAL")
    print("======================================")

    while True:

        print("\n1. Student Registration")
        print("2. Student Login")
        print("3. Exit")

        choice = input("\nEnter your choice (1-3): ")

        if choice == "1":

            register_user()

        elif choice == "2":

            username = user_login()

            if username:
                student_menu(username)

        elif choice == "3":

            print("\nThank you for using our portal!")
            break

        else:

            print("Invalid choice. Please select 1, 2 or 3.")

In [ ]:
main()

 STUDENT PLAGIARISM DETECTION PORTAL

1. Student Registration
2. Student Login
3. Exit



Enter your choice (1-3):  2


File reading operation completed.

--- Student Login ---


Enter your username:  student005
Enter your password:  faststudent5



Login successful!
Welcome, student005

========== STUDENT DASHBOARD ==========
Welcome, student005

1. Submit Assignment
2. Check Assignment Similarity
3. View My Reports
4. Search Assignments
5. Logout



Enter your choice (1-5):  1


... (3 outputs left out)

 Python is a high-level programming language.
 It is widely used for software development.
 we will be doing our python project with python.
 END


File saving operation completed.

Assignment submitted successfully!
Your submission ID is: 6

========== STUDENT DASHBOARD ==========
Welcome, student005

1. Submit Assignment
2. Check Assignment Similarity
3. View My Reports
4. Search Assignments
5. Logout



Enter your choice (1-5):  5


You have been logged out.

1. Student Registration
2. Student Login
3. Exit
